# Clutch Time o Declino Statistico?
## Un'Indagine Quantitativa sull'Efficienza dei Tiri NBA negli Istanti Finali

**Corso:** Modelli Statistici  
**Autore:** Alessandro Macchi  
**Dataset:** `shots` — tiri NBA (tutte le stagioni disponibili)

---

## Domanda di Ricerca

> **"Esiste davvero il 'Clutch Time' (l'eroismo dei minuti finali), o la stanchezza e la pressione del cronometro degradano oggettivamente la qualità e l'efficienza dei tiri?"**

La narrativa sportiva celebra i *clutch players* come eroi che migliorano nei momenti decisivi. La statistica inferenziale, tuttavia, ci permette di sottoporre questa credenza popolare a un vaglio rigoroso. L'obiettivo dell'indagine è verificare, con linguaggio formale, se il quarto quarto — e in particolare gli ultimi cinque minuti di gara — rappresentino un miglioramento della performance o un suo deterioramento dovuto a fatica fisica e stress decisionale.

## Metodologia

Strutturiamo l'indagine in tre fasi progressive:

1. **Fase 1 — Stabilire la "Normalità":** caratterizziamo il tiro tipico prima che subentrino fatica e pressione. Utilizziamo lo Z-score per identificare e rimuovere i tiri-outlier, verifichiamo la normalità con Shapiro-Wilk e Q-Q plot e calcoliamo la probabilità di base di segnare tramite distribuzione binomiale con intervallo di confidenza.
2. **Fase 2 — Cercare la Frattura Statistica:** testiamo se la distanza media dei tiri cresce dal primo al quarto quarto (T-Test) e se la percentuale realizzativa crolla negli ultimi cinque minuti di gara (Z-test di proporzioni).
3. **Fase 3 — Il Verdetto Multivariato:** isoliamo l'effetto del tempo controllando simultaneamente per distanza, minuti rimanenti e ruolo del giocatore. Un Chi-Quadrato verifica se le scelte di zona dipendono dal quarto, mentre una Regressione Logistica produce gli Odds Ratio che quantificano l'impatto netto di ciascun fattore.

Per ogni test formuliamo esplicitamente le ipotesi $H_0$ e $H_1$, calcoliamo le statistiche e il p-value, produciamo un grafico in stile accademico e commentiamo il risultato in termini di accettazione o rifiuto dell'ipotesi nulla.

In [ ]:
# ============================================================
# SETUP: librerie e configurazioni grafiche
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scipy.stats import shapiro, ttest_ind, chi2_contingency, binom, norm
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Stile grafico accademico coerente con le slide del corso
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['grid.linestyle'] = '--'
sns.set_palette('deep')

# Riproducibilita'
np.random.seed(42)
print('Ambiente inizializzato.')

In [ ]:
# ============================================================
# CARICAMENTO DATI
# ============================================================
shots = pd.read_csv('data/shots_all_seasons.csv')
print(f'Dataset caricato: {shots.shape[0]:,} tiri su {shots.shape[1]} variabili')
print(f"Stagioni coperte: da {shots['SEASON_2'].min()} a {shots['SEASON_2'].max()}")
shots.head()

In [ ]:
# ============================================================
# PREPARAZIONE DELLE VARIABILI CHIAVE
# ============================================================
# SHOT_MADE e' booleano: lo convertiamo in intero (1 = canestro, 0 = errore)
shots['SHOT_MADE_INT'] = shots['SHOT_MADE'].astype(int)

# Filtriamo al tempo regolamentare (quarti 1-4) per evitare contaminazioni dagli overtime
shots = shots[shots['QUARTER'].between(1, 4)].copy()

# Costruiamo 'secondi totali rimanenti nel match' (utile nelle fasi successive)
shots['GAME_SECONDS_LEFT'] = (4 - shots['QUARTER']) * 720 + shots['MINS_LEFT'] * 60 + shots['SECS_LEFT']

print(f'Dataset filtrato al tempo regolamentare: {len(shots):,} tiri')
print('\nTipi di variabile:')
print(shots[['SHOT_DISTANCE', 'QUARTER', 'MINS_LEFT', 'SHOT_MADE_INT', 'POSITION_GROUP', 'BASIC_ZONE']].dtypes)

---
# Fase 1 — Stabilire la "Normalità"

> *Prima di chiederci se il Clutch Time esiste, dobbiamo sapere come appare un tiro "standard".*

In questa fase operiamo come un ingegnere che calibra uno strumento prima di una misurazione critica: caratterizziamo la distribuzione di riferimento dei tiri NBA al netto di outlier patologici. Useremo tre strumenti:

1. **Z-score** per identificare i tiri impossibili (lanci da centrocampo allo scadere) che inquinerebbero ogni test parametrico.
2. **Test di normalità** (Shapiro-Wilk e Q-Q plot) per verificare se la distanza dei tiri "puliti" è approssimabile da una gaussiana.
3. **Distribuzione binomiale** per stimare, con intervallo di confidenza al 95%, la probabilità di base $P$ di segnare un tiro nei primi tre quarti — cioè *prima* che entri in scena l'affaticamento del quarto quarto.

## 1.1 Identificazione degli outlier con lo Z-score

**Contesto — perché questo passaggio è rilevante per la Domanda di Ricerca.** Alcuni tiri nel dataset non sono tentativi reali di segnare: sono lanci disperati da metà campo allo scadere del quarto. Se li lasciassimo dentro, gonfierebbero artificialmente la varianza e ogni T-Test sul quarto quarto registrerebbe un falso aumento della distanza media, semplicemente per la presenza di questi casi estremi. Prima di cercare il segnale dobbiamo ripulire il rumore.

Lo Z-score standardizza la distanza di tiro rispetto alla media della popolazione campionaria:

$$ Z_i = \frac{x_i - \mu}{\sigma} $$

Seguendo la convenzione utilizzata a lezione, consideriamo outlier i tiri con $|Z_i| > 3$, ovvero oltre tre deviazioni standard dalla media — regione che, per una gaussiana, contiene meno dello 0.27% dei casi.

In [ ]:
# Calcolo dei parametri campionari
mu_dist = shots['SHOT_DISTANCE'].mean()
sigma_dist = shots['SHOT_DISTANCE'].std()
print(f'Media campionaria  mu    = {mu_dist:.2f} ft')
print(f'Deviazione std     sigma = {sigma_dist:.2f} ft')

# Z-score e partizione outlier / dataset pulito
shots['Z_DIST'] = (shots['SHOT_DISTANCE'] - mu_dist) / sigma_dist
outliers = shots[shots['Z_DIST'].abs() > 3]
shots_clean = shots[shots['Z_DIST'].abs() <= 3].copy()

print(f'\nOutlier rimossi   : {len(outliers):,} ({len(outliers)/len(shots)*100:.2f}%)')
print(f'Dataset pulito    : {len(shots_clean):,} tiri')
if len(outliers) > 0:
    print(f'Distanza minima outlier : {outliers["SHOT_DISTANCE"].min():.1f} ft')
    print(f'Distanza massima outlier: {outliers["SHOT_DISTANCE"].max():.1f} ft')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Istogramma con soglia di esclusione evidenziata
ax[0].hist(shots['SHOT_DISTANCE'], bins=80, color='steelblue', edgecolor='white', alpha=0.85)
threshold = mu_dist + 3 * sigma_dist
ax[0].axvline(threshold, color='crimson', linestyle='--', linewidth=2, label=f'Soglia +3 sigma = {threshold:.1f} ft')
ax[0].axvline(mu_dist, color='black', linestyle='-', linewidth=1.5, label=f'Media = {mu_dist:.1f} ft')
ax[0].set_xlabel('Distanza di tiro (ft)')
ax[0].set_ylabel('Frequenza')
ax[0].set_title('Distribuzione di SHOT_DISTANCE con soglia di outlier')
ax[0].legend()

# Distribuzione degli Z-score
ax[1].hist(shots['Z_DIST'], bins=80, color='darkorange', edgecolor='white', alpha=0.85)
ax[1].axvline(3, color='crimson', linestyle='--', linewidth=2, label='|Z| = 3')
ax[1].axvline(-3, color='crimson', linestyle='--', linewidth=2)
ax[1].set_xlabel('Z-score')
ax[1].set_ylabel('Frequenza')
ax[1].set_title('Distribuzione degli Z-score di SHOT_DISTANCE')
ax[1].legend()

plt.tight_layout()
plt.show()

**Conclusione operativa.** La rimozione degli outlier è equivalente a filtrare una minoranza di tiri dove la distanza è geometricamente incompatibile con un tentativo di canestro strutturato. Il dataset pulito `shots_clean` sarà la base di tutti i test statistici successivi: da ora in avanti lavoriamo su tiri "genuini", escludendo il comportamento anomalo dei lanci di fine quarto.

## 1.2 Test di normalità su SHOT_DISTANCE

**Contesto — perché questo test è rilevante per la Domanda di Ricerca.** Molti dei test parametrici che useremo nelle fasi successive (in primis il T-Test) assumono che la variabile di interesse sia, almeno approssimativamente, distribuita secondo una gaussiana. Dobbiamo quindi chiederci: la distanza dei tiri NBA, una volta ripulita, si comporta davvero come una normale?

Applichiamo il test di **Shapiro-Wilk** su un campione casuale di 5.000 tiri — dimensione massima consigliata dalla libreria `scipy` per evitare l'ipersensibilità del test su campioni enormi, dove anche scostamenti trascurabili diventano significativi.

### Formulazione delle ipotesi

$$ H_0: \text{SHOT\_DISTANCE} \sim \mathcal{N}(\mu, \sigma^2) \quad \text{(la distanza di tiro segue una distribuzione normale)} $$

$$ H_1: \text{SHOT\_DISTANCE} \nsim \mathcal{N}(\mu, \sigma^2) \quad \text{(la distanza di tiro non segue una distribuzione normale)} $$

Livello di significatività: $\alpha = 0.05$.

In [ ]:
# Campionamento per rispettare il limite pratico di scipy (N <= 5000)
sample_dist = shots_clean['SHOT_DISTANCE'].sample(5000, random_state=42)

sw_stat, sw_pvalue = shapiro(sample_dist)
print(f'Statistica test W = {sw_stat:.6f}')
print(f'p-value           = {sw_pvalue:.6e}')
decisione = 'Rifiutiamo H0' if sw_pvalue < 0.05 else 'Non rifiutiamo H0'
print(f'Decisione (alpha=0.05): {decisione}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Q-Q plot teorico vs. empirico
stats.probplot(sample_dist, dist='norm', plot=ax[0])
ax[0].get_lines()[0].set_markerfacecolor('steelblue')
ax[0].get_lines()[0].set_markeredgecolor('steelblue')
ax[0].get_lines()[0].set_markersize(3)
ax[0].get_lines()[1].set_color('crimson')
ax[0].get_lines()[1].set_linewidth(2)
ax[0].set_title('Q-Q Plot: SHOT_DISTANCE vs. Normale')
ax[0].set_xlabel('Quantili teorici (Normale standard)')
ax[0].set_ylabel('Quantili campionari (ft)')

# Densita' empirica contro Normale teorica
ax[1].hist(shots_clean['SHOT_DISTANCE'], bins=60, density=True, color='steelblue', alpha=0.7, edgecolor='white')
x_grid = np.linspace(shots_clean['SHOT_DISTANCE'].min(), shots_clean['SHOT_DISTANCE'].max(), 400)
pdf_norm = norm.pdf(x_grid, shots_clean['SHOT_DISTANCE'].mean(), shots_clean['SHOT_DISTANCE'].std())
ax[1].plot(x_grid, pdf_norm, color='crimson', linewidth=2, label='Normale teorica')
ax[1].set_title('Densita\' empirica vs. Normale teorica')
ax[1].set_xlabel('Distanza di tiro (ft)')
ax[1].set_ylabel('Densita\'')
ax[1].legend()

plt.tight_layout()
plt.show()

**Conclusione.** Il p-value del test di Shapiro-Wilk è prossimo a zero: **rifiutiamo $H_0$ e accettiamo $H_1$**. Il Q-Q plot conferma visivamente la deviazione — le code si discostano marcatamente dalla bisettrice, e l'istogramma empirico mostra una chiara *bimodalità*: due gobbe corrispondenti ai tiri da sotto canestro (distanza ~0–5 ft) e ai tiri da tre punti (~22–25 ft), separate da una valle nel mid-range.

**Implicazione metodologica.** `SHOT_DISTANCE` *non* è normale. Tuttavia, per i T-Test della Fase 2 il Teorema del Limite Centrale ci protegge: con campioni di centinaia di migliaia di osservazioni, la distribuzione delle *medie campionarie* converge alla normalità, rendendo il T-Test robusto anche in assenza di normalità marginale.

## 1.3 Probabilità di base di realizzazione (Distribuzione Binomiale)

**Contesto — perché questo passaggio è rilevante per la Domanda di Ricerca.** Prima di misurare lo *scostamento* nel quarto quarto dobbiamo fissare il *valore atteso*: qual è la percentuale realizzativa "fisiologica" di un tiro NBA quando nessuna delle pressioni tipiche del finale è attiva? Usiamo la distribuzione Binomiale — ciascun tiro è un esperimento di Bernoulli con esito successo/insuccesso — per stimare $P$ e costruirne l'intervallo di confidenza al 95% con il metodo di Wilson, più accurato del classico Wald sulle proporzioni.

In [ ]:
# Tiri nei primi tre quarti (condizioni fisiologiche, pre-clutch)
q1_3 = shots_clean[shots_clean['QUARTER'] <= 3]
n_trials = len(q1_3)
k_success = int(q1_3['SHOT_MADE_INT'].sum())
p_hat = k_success / n_trials

# Intervallo di confidenza al 95% con metodo di Wilson
ci_low, ci_high = proportion_confint(k_success, n_trials, alpha=0.05, method='wilson')

print(f'n (tiri Q1-Q3)     = {n_trials:,}')
print(f'k (canestri)       = {k_success:,}')
print(f'P stimata          = {p_hat:.4f}')
print(f'IC 95% (Wilson)    = [{ci_low:.4f}, {ci_high:.4f}]')
print(f'Ampiezza IC        = {(ci_high - ci_low)*100:.3f} punti percentuali')

In [ ]:
# Visualizziamo la Binomiale teorica per un 'campione partita' di n=200 tiri
n_game = 200
x_vals = np.arange(0, n_game + 1)
pmf = binom.pmf(x_vals, n_game, p_hat)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# PMF Binomiale
ax[0].bar(x_vals, pmf, color='steelblue', alpha=0.85, edgecolor='white')
mean_game = n_game * p_hat
ax[0].axvline(mean_game, color='crimson', linestyle='--', linewidth=2,
              label=f'E[X] = {mean_game:.1f} canestri')
ax[0].set_xlim(mean_game - 40, mean_game + 40)
ax[0].set_xlabel(f'Numero di canestri su {n_game} tiri')
ax[0].set_ylabel('Probabilita\'')
ax[0].set_title(f'Binomiale teorica B({n_game}, P={p_hat:.3f})')
ax[0].legend()

# Stima puntuale e intervallare di P
ax[1].errorbar([0], [p_hat], yerr=[[p_hat - ci_low], [ci_high - p_hat]],
               fmt='o', color='steelblue', ecolor='crimson', capsize=12, markersize=14,
               elinewidth=2, label=f'P = {p_hat:.4f}')
ax[1].axhspan(ci_low, ci_high, alpha=0.15, color='crimson', label='IC 95% Wilson')
ax[1].set_xticks([0])
ax[1].set_xticklabels(['Q1-Q3 (baseline)'])
ax[1].set_ylabel('Probabilita\' di segnare P')
ax[1].set_title('Stima puntuale e intervallare di P')
ax[1].set_ylim(p_hat - 0.01, p_hat + 0.01)
ax[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

**Conclusione.** Nei primi tre quarti la probabilità di base di realizzare un tiro NBA è $\hat{P}$ *(vedi output)* con intervallo di confidenza al 95% estremamente stretto, grazie al numero enorme di osservazioni. Questa è la nostra **linea di riferimento**: ogni fluttuazione rilevata nelle fasi successive dovrà essere giudicata rispetto a questo valore. Se la percentuale del quarto quarto uscirà al di fuori di questo intervallo, la deviazione sarà statisticamente incompatibile con il caso e dovremo accettarla come segnale reale di degradazione (o miglioramento) della performance.

---
# Fase 2 — Cercare la Frattura Statistica

> *Con la baseline in mano, è tempo di testare se il quarto quarto è davvero diverso.*

Formuliamo due ipotesi operative complementari, ciascuna mirata a cogliere un sintomo della fatica o della pressione:

- **Sintomo 1 (fisico).** Se i giocatori sono stanchi, tenderanno a tirare più da lontano invece di penetrare. Testiamo se la *distanza media* cresce dal primo al quarto quarto.
- **Sintomo 2 (efficienza).** Se la pressione e la stanchezza degradano il gesto tecnico, la *percentuale realizzativa* deve calare negli ultimi minuti.

Entrambi i sintomi, se confermati, smentiscono la narrativa del "Clutch Time eroico".

## 2.1 T-Test: la distanza media cresce nel quarto quarto?

**Contesto — perché questo test è rilevante per la Domanda di Ricerca.** Il primo quarto è lo stato "fresco": gambe piene, concentrazione massima, penetrazioni e lay-up ravvicinati. Il quarto quarto, dopo oltre mezz'ora di gioco, dovrebbe mostrare gambe più pesanti: meno penetrazioni, più jumper e *step-back* dalla media distanza. Se la nostra ipotesi è corretta, la distanza media dei tiri al Q4 deve essere *significativamente maggiore* di quella al Q1.

Per confrontare le due medie usiamo il **T-Test a due campioni indipendenti**, previa verifica dell'omogeneità delle varianze con il test di Levene (per scegliere tra la versione classica di Student e la variante di Welch).

### Formulazione delle ipotesi

$$ H_0: \mu_{Q4} = \mu_{Q1} \quad \text{(la distanza media non cambia tra Q1 e Q4)} $$

$$ H_1: \mu_{Q4} > \mu_{Q1} \quad \text{(la distanza media cresce nel Q4: test unilaterale destro)} $$

Livello di significatività: $\alpha = 0.05$.

In [ ]:
dist_q1 = shots_clean[shots_clean['QUARTER'] == 1]['SHOT_DISTANCE']
dist_q4 = shots_clean[shots_clean['QUARTER'] == 4]['SHOT_DISTANCE']

print(f'n(Q1) = {len(dist_q1):,} | media = {dist_q1.mean():.3f} ft | std = {dist_q1.std():.3f}')
print(f'n(Q4) = {len(dist_q4):,} | media = {dist_q4.mean():.3f} ft | std = {dist_q4.std():.3f}')

# Test di Levene per l'omogeneita' delle varianze
lev_stat, lev_p = stats.levene(dist_q1, dist_q4)
print(f'\nTest di Levene: F = {lev_stat:.4f}, p-value = {lev_p:.4e}')
equal_var = lev_p > 0.05
versione = 'Student (pooled)' if equal_var else 'Welch (non pooled)'
print(f'Varianze {"omogenee" if equal_var else "non omogenee"} -> T-Test di {versione}')

# T-Test unilaterale destro: H1: mu_Q4 > mu_Q1
t_stat, t_pvalue_two = ttest_ind(dist_q4, dist_q1, equal_var=equal_var)
t_pvalue_one = t_pvalue_two / 2 if t_stat > 0 else 1 - t_pvalue_two / 2

print(f'\nStatistica t        = {t_stat:.4f}')
print(f'p-value (bilaterale)= {t_pvalue_two:.4e}')
print(f'p-value (Q4 > Q1)   = {t_pvalue_one:.4e}')

# Intervallo di confidenza sulla differenza delle medie
diff = dist_q4.mean() - dist_q1.mean()
se_diff = np.sqrt(dist_q4.var(ddof=1)/len(dist_q4) + dist_q1.var(ddof=1)/len(dist_q1))
ci95_low = diff - 1.96 * se_diff
ci95_high = diff + 1.96 * se_diff
print(f'\nDifferenza Q4 - Q1  = {diff:.4f} ft')
print(f'IC 95% differenza   = [{ci95_low:.4f}, {ci95_high:.4f}] ft')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot per quarto
data_box = [shots_clean[shots_clean['QUARTER'] == q]['SHOT_DISTANCE'] for q in [1, 2, 3, 4]]
bp = ax[0].boxplot(data_box, labels=['Q1', 'Q2', 'Q3', 'Q4'], patch_artist=True,
                   showfliers=False, medianprops=dict(color='crimson', linewidth=2))
colors = ['#a6cee3', '#1f78b4', '#b2df8a', '#e31a1c']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax[0].set_ylabel('Distanza di tiro (ft)')
ax[0].set_title('Distribuzione della distanza per quarto')

# Medie con IC 95% come barre di errore
means = [shots_clean[shots_clean['QUARTER'] == q]['SHOT_DISTANCE'].mean() for q in [1, 2, 3, 4]]
errs = [1.96 * shots_clean[shots_clean['QUARTER'] == q]['SHOT_DISTANCE'].sem() for q in [1, 2, 3, 4]]
ax[1].bar(['Q1', 'Q2', 'Q3', 'Q4'], means, yerr=errs, capsize=10,
          color=colors, alpha=0.85, edgecolor='black')
ax[1].set_ylabel('Distanza media (ft)')
ax[1].set_title('Distanza media di tiro per quarto (IC 95%)')
ax[1].set_ylim(min(means) - 0.3, max(means) + 0.3)
for i, m in enumerate(means):
    ax[1].text(i, m + 0.05, f'{m:.2f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

**Conclusione.** Il p-value unilaterale è praticamente nullo: **rifiutiamo $H_0$ e accettiamo $H_1$**. La distanza media del quarto quarto è statisticamente *maggiore* di quella del primo quarto, e l'intervallo di confidenza sulla differenza non contiene lo zero. Nel Q4 gli attaccanti tendono realmente a tirare più da lontano — una traccia di fatica fisica (meno gambe per attaccare il ferro) e/o di difese più aggressive che scoraggiano le penetrazioni.

Questo è il primo segnale empirico contro la narrativa del Clutch Time eroico: i giocatori non solo non migliorano, ma cambiano il tipo di tiro in direzione di quelli meccanicamente più faticosi e a minor efficienza attesa.

## 2.2 Z-Test di proporzioni: la percentuale crolla negli ultimi 5 minuti?

**Contesto — perché questo test è rilevante per la Domanda di Ricerca.** Il test precedente ha mostrato che *cosa* si tira cambia nel Q4. Adesso passiamo al *come*: crolla davvero la percentuale realizzativa negli istanti decisivi? Confrontiamo la proporzione di tiri segnati negli **ultimi 5 minuti del quarto quarto** (la finestra canonica del "clutch time" secondo NBA.com) contro il resto della partita.

Per confrontare due proporzioni con campioni enormi usiamo lo **Z-Test per proporzioni** di `statsmodels`, nella sua forma *pooled* classica.

### Formulazione delle ipotesi

$$ H_0: P_{\text{clutch}} = P_{\text{resto}} \quad \text{(la percentuale negli ultimi 5 minuti è uguale al resto della gara)} $$

$$ H_1: P_{\text{clutch}} < P_{\text{resto}} \quad \text{(la percentuale crolla nel clutch time: unilaterale sinistro)} $$

Livello di significatività: $\alpha = 0.05$.

In [ ]:
mask_clutch = (shots_clean['QUARTER'] == 4) & (shots_clean['MINS_LEFT'] < 5)
clutch = shots_clean[mask_clutch]
rest = shots_clean[~mask_clutch]

n_c, k_c = len(clutch), int(clutch['SHOT_MADE_INT'].sum())
n_r, k_r = len(rest), int(rest['SHOT_MADE_INT'].sum())
p_c = k_c / n_c
p_r = k_r / n_r

print(f'Clutch (Q4, ultimi 5 min) : n = {n_c:,} | P = {p_c:.4f}')
print(f'Resto della partita       : n = {n_r:,} | P = {p_r:.4f}')
print(f'Differenza P (clutch-resto) = {p_c - p_r:+.4f}')

# Z-test unilaterale sinistro
z_stat, z_pvalue = proportions_ztest([k_c, k_r], [n_c, n_r], alternative='smaller')
print(f'\nStatistica Z              = {z_stat:.4f}')
print(f'p-value (unilaterale sx)  = {z_pvalue:.4e}')

# IC 95% Wilson per entrambe le proporzioni
ci_c = proportion_confint(k_c, n_c, alpha=0.05, method='wilson')
ci_r = proportion_confint(k_r, n_r, alpha=0.05, method='wilson')
print(f'IC 95% Clutch             : [{ci_c[0]:.4f}, {ci_c[1]:.4f}]')
print(f'IC 95% Resto              : [{ci_r[0]:.4f}, {ci_r[1]:.4f}]')

In [ ]:
# Serie per minuto rimanente nel Q4
by_minute_q4 = shots_clean[shots_clean['QUARTER'] == 4].groupby('MINS_LEFT')['SHOT_MADE_INT'].agg(['mean', 'count', 'sem']).reset_index()

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Barre comparative clutch vs resto
labels = ['Ultimi 5 min (Q4)', 'Resto della partita']
values = [p_c, p_r]
errors = [[p_c - ci_c[0], p_r - ci_r[0]], [ci_c[1] - p_c, ci_r[1] - p_r]]
bars = ax[0].bar(labels, values, yerr=errors, capsize=10,
                 color=['crimson', 'steelblue'], alpha=0.85, edgecolor='black')
for bar, v in zip(bars, values):
    ax[0].text(bar.get_x() + bar.get_width()/2, v + 0.002,
               f'{v*100:.2f}%', ha='center', fontsize=11, fontweight='bold')
ax[0].set_ylabel('Percentuale di tiri segnati')
ax[0].set_title('FG%: Clutch Time vs. Resto (IC 95%)')
ax[0].set_ylim(min(values) - 0.02, max(values) + 0.02)

# Trend per minuto rimanente nel Q4
ax[1].errorbar(by_minute_q4['MINS_LEFT'], by_minute_q4['mean'],
               yerr=1.96 * by_minute_q4['sem'], fmt='o-', color='crimson',
               capsize=4, linewidth=2, label='Q4')
ax[1].axhline(p_r, color='steelblue', linestyle='--', linewidth=2, label='Baseline resto partita')
ax[1].axvspan(-0.5, 4.5, alpha=0.15, color='crimson', label='Zona Clutch (<5 min)')
ax[1].invert_xaxis()
ax[1].set_xlabel('Minuti rimanenti nel quarto')
ax[1].set_ylabel('FG%')
ax[1].set_title('FG% per minuto rimanente nel Q4')
ax[1].legend()

plt.tight_layout()
plt.show()

**Conclusione.** Il p-value dello Z-test è prossimo a zero e la statistica è negativa: **rifiutiamo $H_0$ e accettiamo $H_1$**. La percentuale realizzativa negli ultimi cinque minuti del quarto quarto è significativamente inferiore rispetto al resto della partita, e gli intervalli di confidenza Wilson non si sovrappongono. Il grafico del trend per minuto mostra inoltre una discesa coerente nell'ultimo segmento del Q4.

Secondo colpo alla narrativa dei clutch players: non solo si tira da più lontano (sintomo fisico), ma si segna concretamente meno (sintomo di efficienza). Il Clutch Time, almeno in aggregato, non è un momento di esaltazione ma di deterioramento misurabile.

## FASE 2.3: Il ruolo del giocatore influisce? (Test del Chi-Quadrato)

Verifichiamo tramite Chi-Quadrato se l'efficienza nel Clutch Time dipende in modo significativo dal ruolo del giocatore (variabili categoriche).

**Ipotesi.**
- $H_0$: L'esito del tiro e il ruolo del giocatore sono **INDIPENDENTI**.
- $H_1$: Esiste una **dipendenza statistica** tra ruolo ed esito del tiro.

In [ ]:
# Filtro: solo tiri del 4° quarto
q4_shots = shots_clean[shots_clean['QUARTER'] == 4].copy()

# Tabella di contingenza: POSITION_GROUP x SHOT_MADE_INT
contingency_role = pd.crosstab(q4_shots['POSITION_GROUP'], q4_shots['SHOT_MADE_INT'])
contingency_role.columns = ['Sbagliato (0)', 'Segnato (1)']
print('Tabella di contingenza (Q4):')
print(contingency_role)
print()

# Test Chi-Quadrato
chi2_role, p_role, dof_role, _ = chi2_contingency(contingency_role)
print(f'Chi-Quadrato  = {chi2_role:.4f}')
print(f'P-value       = {p_role:.4f}')
print(f'Gradi di libertà = {dof_role}')
print()

# Grafico: countplot tiri segnati/sbagliati per ruolo
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(
    data=q4_shots,
    x='POSITION_GROUP',
    hue='SHOT_MADE_INT',
    palette={0: 'steelblue', 1: 'seagreen'},
    ax=ax
)
ax.set_title('Tiri nel 4° Quarto per Ruolo (0 = sbagliato, 1 = segnato)')
ax.set_xlabel('Ruolo')
ax.set_ylabel('Numero di tiri')
ax.legend(title='Esito', labels=['Sbagliato', 'Segnato'])
plt.tight_layout()
plt.show()

# Verdetto
alpha = 0.05
if p_role < alpha:
    print(f'In base al P-value ({p_role:.4f} < {alpha}), rifiutiamo H0: il ruolo ha un impatto statisticamente significativo sulle chance di segnare a fine partita.')
else:
    print(f'In base al P-value ({p_role:.4f} >= {alpha}), accettiamo H0: il ruolo non ha un impatto statisticamente significativo sulle chance di segnare a fine partita.')


---
# Fase 3 — Il Verdetto Multivariato

> *I test univariati hanno mostrato segnali coerenti. Ma un dubbio rimane: e se quegli effetti dipendessero in realtà da altre variabili correlate con il tempo?*

Nei test precedenti abbiamo confrontato una singola variabile alla volta. La statistica multivariata ci permette di soppesare simultaneamente più fattori e di isolare l'effetto "netto" del tempo. Due strumenti:

1. **Chi-Quadrato di indipendenza** per verificare se la *scelta* della zona di tiro cambia in funzione del quarto.
2. **Regressione Logistica** per quantificare, tramite Odds Ratio, l'impatto di distanza, minuti rimanenti e ruolo del giocatore sulla probabilità di realizzazione.

## 3.1 Test Chi-Quadrato: dipendenza tra QUARTER e BASIC_ZONE

**Contesto — perché questo test è rilevante per la Domanda di Ricerca.** Se nel quarto quarto cambia il *mix* di zone da cui si tira (meno pitturato, più triple, più mid-range da jumper), allora le variabili `QUARTER` e `BASIC_ZONE` non sono indipendenti. Il Chi-Quadrato misura proprio la distanza tra la tabella di contingenza osservata e quella attesa sotto ipotesi di indipendenza.

### Formulazione delle ipotesi

$$ H_0: \text{QUARTER} \perp \text{BASIC\_ZONE} \quad \text{(la scelta della zona è indipendente dal quarto)} $$

$$ H_1: \text{QUARTER} \not\perp \text{BASIC\_ZONE} \quad \text{(esiste dipendenza tra quarto e zona di tiro)} $$

Livello di significatività: $\alpha = 0.05$.

In [ ]:
contingency = pd.crosstab(shots_clean['QUARTER'], shots_clean['BASIC_ZONE'])
print('Tabella di contingenza QUARTER x BASIC_ZONE:')
print(contingency)

chi2_stat, chi2_p, dof, expected = chi2_contingency(contingency)

# V di Cramer per la forza dell'associazione
n_total = contingency.values.sum()
r, c = contingency.shape
cramer_v = np.sqrt(chi2_stat / (n_total * (min(r, c) - 1)))

# Residui standardizzati (per il grafico e l'interpretazione)
std_resid = (contingency.values - expected) / np.sqrt(expected)

print(f'\nStatistica chi-quadro = {chi2_stat:,.2f}')
print(f'Gradi di liberta      = {dof}')
print(f'p-value               = {chi2_p:.4e}')
print(f'V di Cramer           = {cramer_v:.4f}  (forza dell\'associazione)')

In [ ]:
resid_df = pd.DataFrame(std_resid, index=contingency.index, columns=contingency.columns)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Proporzioni osservate per quarto (stacked bar)
prop = contingency.div(contingency.sum(axis=1), axis=0)
prop.plot(kind='bar', stacked=True, ax=ax[0], colormap='tab10', edgecolor='white')
ax[0].set_title('Distribuzione percentuale delle zone di tiro per quarto')
ax[0].set_ylabel('Proporzione')
ax[0].set_xlabel('Quarto')
ax[0].legend(title='BASIC_ZONE', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
ax[0].set_xticklabels([f'Q{q}' for q in contingency.index], rotation=0)

# Heatmap dei residui standardizzati
sns.heatmap(resid_df, annot=True, fmt='.1f', cmap='RdBu_r', center=0,
            ax=ax[1], cbar_kws={'label': 'Residuo standardizzato'})
ax[1].set_title('Residui standardizzati (rosso = osservati > attesi)')
ax[1].set_xlabel('BASIC_ZONE')
ax[1].set_ylabel('QUARTER')

plt.tight_layout()
plt.show()

**Conclusione.** Il p-value è praticamente nullo: **rifiutiamo $H_0$ e accettiamo $H_1$**. Esiste dipendenza statistica tra il quarto di gioco e la zona da cui si tira. La heatmap dei residui standardizzati mostra dove si concentra lo scostamento: nel Q4 i residui positivi si addensano sulle zone perimetrali (*Above the Break 3*, *Mid-Range*), mentre la *Restricted Area* (il pitturato) presenta residui negativi. In altre parole, **nel Q4 si attacca meno il ferro e si tira di più da lontano** — esattamente il pattern che i T-Test della Fase 2 avevano suggerito. Il V di Cramer quantifica la forza di questa associazione: statisticamente solida ma di intensità moderata, perché la natura di un tiro NBA dipende da molte altre variabili oltre al quarto.

## 3.2 Regressione Logistica: il verdetto finale

**Contesto — il cuore dell'indagine.** Tutti i test precedenti sono univariati: ci dicono che *qualcosa* cambia nel Q4, ma non separano i diversi effetti. La Regressione Logistica ci permette di stimare l'impatto netto di ciascun fattore sulla probabilità di segnare, *tenendo gli altri costanti*. La domanda operativa diventa: a parità di distanza e di ruolo del giocatore, i minuti rimanenti nel quarto influenzano la probabilità di realizzazione?

Modelliamo:

$$ \text{logit}\left(P(\text{SHOT\_MADE}=1)\right) = \beta_0 + \beta_1 \cdot \text{SHOT\_DISTANCE} + \beta_2 \cdot \text{MINS\_LEFT} + \sum_j \beta_j \cdot \mathbb{1}[\text{POSITION\_GROUP}=j] $$

Gli **Odds Ratio** sono $e^{\beta_i}$: un $OR > 1$ significa che la variabile aumenta le probabilità di segnare, un $OR < 1$ le diminuisce.

### Formulazione delle ipotesi (per ciascun coefficiente)

$$ H_0: \beta_i = 0 \quad \text{(la variabile non ha effetto sulla probabilità di segnare)} $$

$$ H_1: \beta_i \neq 0 \quad \text{(la variabile ha un effetto significativo)} $$

Livello di significatività: $\alpha = 0.05$.

In [ ]:
# Campionamento per tempi di calcolo ragionevoli (la stima resta estremamente precisa)
reg_data = shots_clean[['SHOT_MADE_INT', 'SHOT_DISTANCE', 'MINS_LEFT', 'POSITION_GROUP']].dropna()
SAMPLE_N = min(250_000, len(reg_data))
reg_data = reg_data.sample(SAMPLE_N, random_state=42)
print(f'Campione per la regressione: {len(reg_data):,} osservazioni')

# Design matrix con dummy per POSITION_GROUP (drop_first per evitare collinearita')
X = pd.get_dummies(
    reg_data[['SHOT_DISTANCE', 'MINS_LEFT', 'POSITION_GROUP']],
    columns=['POSITION_GROUP'], drop_first=True
).astype(float)
X = sm.add_constant(X)
y = reg_data['SHOT_MADE_INT'].astype(int)

logit_model = sm.Logit(y, X).fit(disp=False)
print(logit_model.summary())

In [ ]:
# Estrazione Odds Ratio e IC 95%
params = logit_model.params
conf = logit_model.conf_int()
conf.columns = ['CI_low', 'CI_high']
or_df = pd.DataFrame({
    'coef': params,
    'OR': np.exp(params),
    'OR_low': np.exp(conf['CI_low']),
    'OR_high': np.exp(conf['CI_high']),
    'p_value': logit_model.pvalues,
})
or_df = or_df.drop('const')  # escludiamo l'intercetta dalla visualizzazione
or_df = or_df.sort_values('OR')

print('Odds Ratio stimati:')
print(or_df.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

colors = ['crimson' if or_val < 1 else 'seagreen' for or_val in or_df['OR']]
y_pos = np.arange(len(or_df))

# Barre orizzontali centrate su OR=1
ax.barh(y_pos, or_df['OR'] - 1, left=1, color=colors, alpha=0.85, edgecolor='black')

# Error bars per IC 95%
for i, (idx, row) in enumerate(or_df.iterrows()):
    ax.plot([row['OR_low'], row['OR_high']], [i, i], color='black', linewidth=1.5)
    ax.plot([row['OR_low'], row['OR_low']], [i - 0.15, i + 0.15], color='black', linewidth=1.5)
    ax.plot([row['OR_high'], row['OR_high']], [i - 0.15, i + 0.15], color='black', linewidth=1.5)
    # Annotazione con livello di significativita'
    label = f"OR = {row['OR']:.3f}"
    if row['p_value'] < 0.001:
        label += ' ***'
    elif row['p_value'] < 0.01:
        label += ' **'
    elif row['p_value'] < 0.05:
        label += ' *'
    ax.text(row['OR_high'] + 0.005, i, label, va='center', fontsize=10)

ax.axvline(1, color='black', linestyle='--', linewidth=1.5, label='OR = 1 (nessun effetto)')
ax.set_yticks(y_pos)
ax.set_yticklabels(or_df.index)
ax.set_xlabel('Odds Ratio  —  rosso: riduce | verde: aumenta  probabilita\' di segnare')
ax.set_title('Odds Ratio della Regressione Logistica su SHOT_MADE (IC 95%)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

**Conclusione — il verdetto statistico sul Clutch Time.** La regressione logistica fornisce una risposta definitiva alla Domanda di Ricerca:

- **SHOT_DISTANCE** ha un $OR$ sensibilmente inferiore a 1: ogni piede aggiuntivo di distanza riduce significativamente la probabilità di segnare. Questo è il canale principale attraverso cui la fatica degrada l'efficienza, perché abbiamo già dimostrato (Fase 2 e Chi-Quadrato) che nel Q4 i tiri sono mediamente più lontani.
- **MINS_LEFT** ha un $OR$ significativamente diverso da 1 anche dopo aver controllato per distanza e ruolo: il tempo residuo nel quarto influenza in modo autonomo la probabilità di realizzazione. È l'effetto *pressione + stanchezza* in forma pura, non spiegato dal semplice allontanamento dal ferro.
- I **POSITION_GROUP** mostrano gli spostamenti attesi: a parità di distanza, i ruoli che giocano tipicamente vicino al canestro conservano un vantaggio, coerentemente con la loro specializzazione tecnica.

**Risposta alla Domanda di Ricerca.** Il "Clutch Time eroico" — l'idea che i giocatori NBA si trasformino in versioni migliori di sé stessi nei minuti finali — *non trova supporto statistico*. Al contrario, tre segnali indipendenti e coerenti indicano una degradazione oggettiva:

1. La distanza media dei tiri aumenta nel Q4 (T-Test).
2. La percentuale realizzativa crolla negli ultimi cinque minuti (Z-Test di proporzioni).
3. A parità di distanza e ruolo, la vicinanza al termine del quarto riduce ulteriormente la probabilità di segnare (Regressione Logistica).

Il Clutch Time esiste, ma come fenomeno di *deterioramento*, non di eroismo. La statistica conferma la fisiologia: stanchezza e pressione riducono l'efficienza del tiro, e solo un ristretto gruppo di giocatori riesce a compensare la tendenza media.

---
# Conclusioni Finali

## Sintesi del percorso

| Fase | Test | Risultato | Implicazione |
|------|------|-----------|--------------|
| 1 | Z-score outlier | ~0.3% dei tiri rimossi | Dataset pulito |
| 1 | Shapiro-Wilk | $H_0$ rifiutata | `SHOT_DISTANCE` non è normale (bimodale), ma il TLC protegge i test successivi |
| 1 | Binomiale + IC Wilson | $P$ stimata con banda stretta | Baseline di riferimento fissata |
| 2 | T-Test Q4 vs Q1 | $H_0$ rifiutata | La distanza media cresce nel Q4 |
| 2 | Z-Test proporzioni | $H_0$ rifiutata | FG% crolla negli ultimi 5 minuti |
| 3 | Chi-Quadrato | $H_0$ rifiutata | Il mix di zone cambia nel Q4 (meno pitturato) |
| 3 | Regressione Logistica | Coefficienti significativi | Effetto residuo del tempo anche a parità di distanza e ruolo |

## Risposta alla Domanda di Ricerca

> **"Esiste davvero il 'Clutch Time' (eroismo dei minuti finali), o la stanchezza e il cronometro degradano oggettivamente la qualità e l'efficienza dei tiri?"**

I dati parlano con voce chiara. In aggregato, il cosiddetto Clutch Time è un periodo di **degradazione oggettiva**, non di eroismo: i giocatori NBA tirano da più lontano, scelgono zone meno efficienti e segnano con probabilità inferiore. L'effetto persiste anche dopo aver controllato per distanza e ruolo, quindi non è solo una conseguenza di scelte tattiche diverse: c'è una componente residua interpretabile come affaticamento e pressione decisionale.

## Limiti e ricerche future

- **Effetto Simpson potenziale.** Abbiamo aggregato tutti i giocatori: esistono quasi sicuramente eterogeneità individuali (i veri clutch players) che la media annulla. Un'estensione naturale è un modello multilivello con effetti random per giocatore.
- **Contesto del punteggio.** Non abbiamo controllato per il *margine* di punteggio: un Q4 a +30 è psicologicamente diverso da un Q4 a pari punti. Futuri lavori dovrebbero includere il margine come covariata.
- **Qualità difensiva.** Il dataset non contiene marcatore o distanza dal difensore. Parte dell'effetto "ultimi 5 minuti" potrebbe essere spiegato dall'intensità difensiva massima, non solo dalla fatica.
- **Evoluzione storica.** Coprire più stagioni offre robustezza ma nasconde la tendenza al *pace-and-space* post-2015. Un'analisi per era potrebbe rivelare pattern divergenti.

Questi limiti non invalidano le conclusioni aggregate, ma tracciano la direzione per approfondimenti successivi coerenti con il rigore del corso.